Your report should read like a professional paper that you would publish for a research project or an internship/job. For this reason, you need to follow some formatting rules:

- Everything required in the header cell should be properly entered.
- All code in the report should be "folded". (Note that `code-fold: true` in the header cell already does this.)
- All the sections included below this cell should be present in the report (except for Appendix, if not used).
- The indicated cells (including this one) should be deleted from the final report.
- The description for each section below this cell should also be deleted from the final report.
- No irrelevant code/text/output/figure should be present in the report.
- **3 points will be taken off PER formatting rule that is not followed.**

**Language:** The language of the report should be professional and always consider both the technical and the real-life aspects of your analysis and results. **If the language is found inadequate, 5 points will be taken off.** You can check with your instructor if you have any questions.

**This cell is only informational and should not be kept in the report; you need to delete it while preparing the report using the template.** 

## 1) Introduction

Social media posts rarely carry meaning through a single channel. A tweet is often a short caption and an image, and the two together convey a mood that neither might fully express alone. This project asks a focused version of that observation as a machine-learning question: for a tweet that comes with both a caption and an image, which representation carries the most predictable sentiment, the text, the image, or the two combined? To answer it, three parallel classification pipelines are built on the same tweets and compared on identical terms (text-only, image-only, and a multimodal pipeline that fuses the two), each predicting one of three sentiment classes (negative, neutral, positive).

Sentiment analysis is one of the most widely deployed tasks in machine learning, from brand monitoring to content moderation (Pang & Lee, 2008). On the text side, the field has moved from hand-engineered features such as TF-IDF with classical classifiers toward learned representations and pretrained word embeddings (Pennington, Socher, & Manning, 2014). On the image side, convolutional networks and transfer learning from ImageNet-pretrained models are the standard approach for small datasets (Sandler et al., 2018; Tan & Le, 2019). The newer question is multimodal learning, how to combine signals from different data types, now an active research area in its own right (Baltrušaitis, Ahuja, & Morency, 2019). This project is a small, controlled instance of that problem, spanning classical models, recurrent networks, transfer learning, and an ensemble fusion step.

I chose this topic because it forces a question that matters well beyond this dataset: when is more data actually more information? It is tempting to assume a second modality can only help, but that assumption deserves to be tested rather than asserted. A comparative design also teaches something more durable than any single model, namely how to build an evaluation that reveals when an idea did not work, and why.

## 2) Data

### Source

The data is the Twitter Dataset for Sentiment Analysis, published by Dunya Jasim on Kaggle (Jasim, n.d.). It contains 4,869 real tweets, and its defining feature for this project is that every tweet is paired: each carries both a text caption and a corresponding image, labeled with a single sentiment class. This pairing is what makes a controlled comparison across modalities possible, because the text and the image describe the same post and share the same ground-truth label.

### Format and modalities

The dataset combines two input modalities with one categorical target. The text lives in `LabeledText.csv` (read with `latin-1` encoding), which has three columns: `File Name`, `Caption`, and `LABEL`. The images live in `data/Images/`, split across three class subfolders. Text and images are linked by a numeric identifier embedded in the file name, so that `1.txt` in the CSV corresponds to the image `1.jpg` in the appropriate class folder.

The two modalities have very different properties:

- *Text.* The captions are short and informal, averaging about 13 words, and contain the usual social-media noise of URLs, hashtags, mentions, and inconsistent capitalization. Text is a one-dimensional sequence of tokens whose meaning depends on word identity and order.
- *Image.* The images are variable-resolution JPEG files and are visually heterogeneous. They are a mix of ordinary photographs, memes with text overlaid on them, and Snapchat-style screenshots. An image is a three-dimensional array of pixel intensities (height, width, and three color channels), with no inherent alignment to the words in the caption.

The target is a three-class sentiment label. The classes are mildly imbalanced, as shown in @tbl-labels.

| Class | Count | Share |
|---|---|---|
| Neutral | 1,771 | 36.4% |
| Positive | 1,646 | 33.8% |
| Negative | 1,452 | 29.8% |
| Total | 4,869 | 100% |

: Class distribution of the 4,869 tweets. {#tbl-labels}

One paired example per class is shown in @fig-examples, with each image beside its caption. The pairing is what the multimodal pipeline later exploits, and the figure also previews a difficulty that shapes the entire image analysis: the sentiment of a tweet is often carried by the caption, while the image alone can be ambiguous or even unrelated to the mood of the post.

### Why this dataset

This dataset was chosen specifically because it is paired. Most sentiment datasets are single-modality, which makes any text-versus-image comparison confounded by the fact that the two would describe different posts. Here, because each text and image refer to the same tweet and share a label, the comparison isolates the modality itself as the variable of interest. The dataset is also a convenient size: at roughly 4,900 examples it is small enough to train every model on a personal machine, yet large enough that overfitting and data-size effects become real concerns worth studying, which is itself instructive.

### Non-modeling preprocessing

Two kinds of preprocessing are applied before any model sees the data, and neither uses information from the labels or the test set.

For text, each caption is lowercased, stripped of URLs and any non-alphabetic characters, tokenized, and filtered to remove English stopwords and single-character tokens. This cleaning is purely rule-based. A side effect worth noting is that 102 captions reduce to empty strings, because they consisted entirely of links, punctuation, or stopwords. These rows are retained rather than dropped, so that the text and image pipelines operate on the same set of tweets.

For images, each file is decoded to three color channels and resized to a fixed 128 by 128 pixels using Lanczos resampling, since the models require a uniform input shape. Pixel values are then rescaled, either to the unit interval for the from-scratch network or to the range expected by each pretrained model. The resolution of 128 pixels is a deliberate compromise: it preserves enough detail for photograph-style images while keeping memory and training time manageable. It is worth stating plainly that no feasible resolution would let a convolutional network read the small overlaid caption text in the meme-style images, a point returned to in the discussion.

Finally, the data is divided with a stratified split using a fixed random seed, holding out 20% as a test set and reserving a further portion of the training data for validation. Stratification preserves the class proportions in every split, and the fixed seed makes the evaluation reproducible. The exact splitting protocol, including how it differs between the single-modality and multimodal pipelines, is described in the Methods section.

In [ ]:
#| label: fig-examples
#| fig-cap: "Paired tweet examples: one image and its caption per sentiment class. The caption frequently carries the sentiment more clearly than the image."
import os
import textwrap
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

meta = pd.read_csv('data/LabeledText.csv', encoding='latin-1')
meta['file_id'] = meta['File Name'].str.replace('.txt', '', regex=False)
label_to_folder = {'negative': 'Negative', 'neutral': 'Neutral', 'positive': 'positive'}

# One paired example per class, chosen for a readable, moderate-length caption
fig, axes = plt.subplots(3, 2, figsize=(10, 7.5), gridspec_kw={'width_ratios': [1, 1.25]})
for r, label in enumerate(['negative', 'neutral', 'positive']):
    sub = meta[(meta['LABEL'].str.lower() == label) &
               (meta['Caption'].astype(str).str.split().apply(len).between(6, 18))]
    row = sub.sample(1, random_state=42).iloc[0]
    img_path = os.path.join('data/Images', label_to_folder[label], f"{row['file_id']}.jpg")

    img = Image.open(img_path).convert('RGB')
    axes[r][0].imshow(img)
    axes[r][0].set_xticks([])
    axes[r][0].set_yticks([])
    axes[r][0].set_ylabel(label.capitalize(), fontsize=12)

    axes[r][1].axis('off')
    caption = textwrap.fill(str(row['Caption']).strip(), width=38)
    axes[r][1].text(0.0, 0.5, caption, va='center', ha='left', fontsize=10.5)

plt.tight_layout()
plt.show()

## 3) Methods

### Task and variables

The task is supervised multiclass classification. The response variable is the sentiment label, which takes one of three values (negative, neutral, positive). The predictors differ by pipeline: the text pipeline predicts from the caption alone, the image pipeline predicts from the image alone, and the multimodal pipeline predicts from both. Because the same labeled tweets feed all three pipelines, any difference in performance can be attributed to the representation rather than to the data.

### Evaluation protocol and metrics

Every model is held to the same protocol so that results are comparable. The data is split once into a training portion and a test portion, with 20% held out for testing, stratified by class and fixed with a random seed so the test set never changes. For models with hyperparameters to tune, a validation set is carved from the training portion using the same 20% proportion and seed. Hyperparameters are selected on the validation set, after which the chosen model is retrained on the combined training and validation data before a single, final evaluation on the test set. A single held-out validation set is used rather than k-fold cross-validation, partly for consistency across the classical and deep models so they are judged identically, and partly because the deep networks are too costly to train many times over. The cost of this choice is a noisier estimate of validation performance, an issue that becomes relevant when tuning the fusion weight later.

Two metrics summarize performance: accuracy and the macro-averaged F1 score. Accuracy is the fraction of test tweets classified correctly. On its own accuracy can be misleading under class imbalance, because a model can score reasonably well simply by favoring the largest class, so the macro F1 score is reported alongside it. For each class, precision is the share of predictions for that class that are correct, recall is the share of true members of that class that are recovered, and the F1 score is their harmonic mean, which is high only when both are high. The macro F1 score averages the per-class F1 scores with equal weight, so a model that ignores a minority class is penalized regardless of how common that class is. Per-class F1 scores are also examined directly, because they reveal failure modes that a single summary number hides, such as a model that predicts only the majority class. As a reference point, every comparison includes a majority-class baseline that always predicts the most common label (neutral); any model that cannot beat it has learned nothing useful.

### Text pipeline

The text pipeline spans two families of model that differ in how they represent a caption. The classical models use a term frequency-inverse document frequency (TF-IDF) representation, which turns each caption into a sparse vector over the training vocabulary, weighting each word by how often it appears in the caption and downweighting words that are common across all captions. This is a form of explicit feature engineering, and the vocabulary and weights are learned from the training data only, so no information leaks from the test set. On top of these features three standard classifiers are trained. Multinomial naive Bayes treats the features as word counts and applies Bayes' rule under the simplifying assumption that words are conditionally independent given the class. Logistic regression fits a linear decision boundary by modeling class log-odds as a weighted sum of the features, with an L2 penalty whose strength is set by the regularization parameter C. A linear support vector machine separates classes by the hyperplane that maximizes the margin between them, again tuned through C (Cortes & Vapnik, 1995). A multilayer perceptron, a small fully connected neural network, is also trained on the same features as a nonlinear classical comparison.

The neural models instead represent a caption as a sequence. Each caption is tokenized and padded to a fixed length, and each token is mapped to a dense vector. The first neural model is a plain recurrent neural network (RNN), which reads the sequence one token at a time while maintaining a hidden state, and learns its word vectors from scratch. The second replaces the recurrent unit with a long short-term memory (LSTM) cell, which adds gating mechanisms that let it retain or forget information across longer spans and largely avoids the vanishing-gradient problem of plain RNNs (Hochreiter & Schmidhuber, 1997). Crucially, the LSTM's word vectors are initialized from GloVe, a set of 100-dimensional embeddings pretrained on six billion tokens of general text that place semantically related words near one another (Pennington, Socher, & Manning, 2014). These embeddings are held fixed during training, so the model inherits a useful notion of word meaning rather than having to learn one from a few thousand short captions. The full set of text models is summarized in @tbl-text-models.

| Model | Family | Representation | Tuned hyperparameter |
|---|---|---|---|
| Baseline | Reference | none (predicts majority class) | none |
| Multinomial Naive Bayes | Classical | TF-IDF | none |
| Logistic Regression | Classical | TF-IDF | C (L2 strength) |
| Linear SVM | Classical | TF-IDF | C (margin) |
| Multilayer Perceptron | Neural (dense) | TF-IDF | none |
| RNN (plain) | Neural (recurrent) | embeddings learned from scratch | hidden units |
| GloVe + LSTM | Neural (recurrent) | frozen GloVe embeddings | LSTM units |

: Models in the text pipeline. {#tbl-text-models}

### Image pipeline

The image pipeline takes the 128 by 128 pixel arrays as input and likewise spans a from-scratch model and transfer-learning models. The from-scratch model is a convolutional neural network (CNN). A CNN slides small learnable filters across the image to detect local patterns such as edges and textures, stacks several such convolutional layers so that later layers compose simple patterns into more complex ones, and uses pooling layers to progressively reduce spatial resolution while retaining the strongest responses. The network used here applies three convolutional blocks followed by fully connected layers that map the learned features to the three sentiment classes. Because a network trained from scratch must learn all of its visual features from only a few thousand images, it is prone to either overfitting or collapse on a dataset this small.

The transfer-learning models address that limitation by reusing visual features already learned on a much larger corpus. Both MobileNetV2 (Sandler et al., 2018) and EfficientNetB0 (Tan & Le, 2019) are convolutional networks pretrained on ImageNet, a dataset of more than a million labeled photographs (Deng et al., 2009). For this project their pretrained convolutional layers are frozen and used as a fixed feature extractor, a global average pooling layer condenses the spatial feature maps into a single vector, and a small trainable classification head is added on top. The head consists of a dense layer with L2 regularization, a dropout layer that randomly zeros half of its inputs during training to discourage co-adaptation and overfitting (Srivastava et al., 2014), and a final layer producing class probabilities. All neural models are trained with the Adam optimizer at a low learning rate (Kingma & Ba, 2015) and use early stopping, which halts training when validation loss stops improving. The image models are summarized in @tbl-image-models. One honest methodological detail is worth recording: the two transfer models differ in their final-retraining early-stopping signal. MobileNetV2 reused the fixed validation set, which with so few images it memorized, so early stopping never triggered; EfficientNetB0 was therefore switched to carve a fresh validation slice from its training data at fit time, giving early stopping a signal the model had not already seen. A logistic-regression classifier on raw flattened pixels was also attempted, but it was abandoned because fitting on roughly fifty thousand dense pixel features per image was prohibitively slow, which itself illustrates why classical models are ill-suited to raw image data.

| Model | Approach | Feature source | Tuned hyperparameter |
|---|---|---|---|
| Baseline | Reference | none (predicts majority class) | none |
| CNN (scratch) | Trained from scratch | features learned from pixels | convolutional filter count |
| MobileNetV2 | Transfer learning | frozen ImageNet backbone | dense head size |
| EfficientNetB0 | Transfer learning | frozen ImageNet backbone | dense head size |

: Models in the image pipeline. {#tbl-image-models}

### Multimodal pipeline

The multimodal pipeline combines the strongest text model (the GloVe-initialized LSTM) and the strongest image model (MobileNetV2) by late fusion, meaning each model first produces its own class-probability vector for a tweet and the two vectors are then combined into a single prediction. Late fusion is a natural fit here because the two modalities have incompatible raw forms but produce comparable outputs, namely a probability over the same three classes. To keep the comparison honest, the multimodal pipeline does not reuse the separate text and image test sets; instead it builds one combined table that joins each caption to its image by numeric identifier and performs a single joint split, so that both base models are trained and evaluated on exactly the same tweets.

Three fusion strategies are compared, in increasing order of flexibility, and are summarized in @tbl-fusion. The first is an equal average of the two probability vectors. The second is a weighted average that assigns weight to the text model and the remainder to the image model, with the weight chosen on the validation set. The third is stacking, in which the two probability vectors are concatenated into a six-dimensional feature vector and a logistic-regression meta-classifier is trained to map it to the final prediction, allowing each modality's per-class probabilities to be weighted independently rather than through a single shared coefficient (Wolpert, 1992). For the weighted and stacking strategies the fusion parameters must be fit on data the base models have not already seen, so for these the base models are trained on the training portion alone and the validation set is held back to tune the weight or fit the meta-classifier. Training the base models on the full training-plus-validation data instead would let them memorize the validation set and make any tuning on it meaningless. All implementations use scikit-learn for the classical models and metrics (Pedregosa et al., 2011) and TensorFlow with Keras for the neural networks (Abadi et al., 2016).

| Strategy | How the two probability vectors are combined | Parameters fit on |
|---|---|---|
| Equal average | mean of the text and image probabilities | none |
| Weighted average | weight on text, remainder on image | fusion weight tuned on validation |
| Stacking | logistic regression on the concatenated probabilities | meta-classifier fit on validation |

: Late-fusion strategies in the multimodal pipeline. {#tbl-fusion}

## 4) Results and Discussion

- Describe how your model(s) is/are applied to your dataset, with the emphasis on handling different modalities in the dataset.
- Clearly display your results. Both numeric and visual results must be displayed **clearly and in a professional way**. That includes (but is not limited to) tabulating numbers, labeling graphs, using proper units, etc.
- Interpret your results and **justify your interpretations**:
    - Can your ML workflow be considered successful or not? If yes, why?
    - If not successful, what are the possible reasons for that? How could these reasons be accounted for and why are they not addressed in this project?
- Explain how your results demonstrate the important functionalities of your model(s).

**(30 points)**

## 5) Conclusion

Include a brief conclusion that summarizes your findings and connects them to the ML tools you covered in this report. **(2.5 points)**

## References

If you use any bibliographical references in your report, cite them in the text and list the references here. Any citation style is acceptable. **(No points allotted for this section, but 15 points will be taken off for inadequate (or non-existent) citations.)**

## Appendix

If there is any supporting/additional material that should not be in the main report, but still carries some relevance to the report, you may include it here. **If you do not have any such material, remove this section.**